# Lab 3: give each agent its own identity (Entra Agent ID + LangGraph)

**What you'll build:** a three-agent team in LangGraph. A *supervisor* sends work to a *researcher* and then to a *writer*.
Each worker signs in to Microsoft Graph as **its own Entra Agent ID**.

**Why it matters:** many multi-agent apps run every agent under **one** app registration and **one** secret. Then:
* every agent has **every permission** the app has, even agents that only write text;
* the logs say "the app did it", **not which agent**;
* you can't switch off one misbehaving agent without stopping all of them.

With Entra Agent ID, each agent gets its own identity, so it gets **its own permissions**, **its own entries in the sign-in
logs**, and can be **disabled on its own**.

**What you'll see at the end:** the same Graph request ("list users") **succeeds for the researcher** and is **refused (403)
for the writer**, because only the researcher was given permission.

### Words you need

| Term | What it means |
|---|---|
| **Blueprint** | A special Entra application object for an agent type. It holds a credential or trusts a managed identity through federation. One blueprint can create many agent identities. |
| **Agent identity** | The security identity of one agent, e.g. "researcher". It is a **service principal**, not a separate ordinary app registration, and cannot hold its own secret. The runtime uses the blueprint to obtain its tokens. |
| **Access token** | A short-lived, signed pass for a particular API (`aud`). It identifies the caller (`appid` or `azp`, depending on token version) and can carry application permissions (`roles`) or delegated scopes (`scp`). It is not an LLM text token. |
| **Permission (app role)** | E.g. `User.Read.All` = "may read every user profile in the directory". Granted **per agent identity**. |
| **LangGraph** | A Python library for multi-agent workflows. **Nodes** are the agents; **edges** say who goes next; the **state** is a shared dictionary the agents read and write. |

### What is real and what is fake?

| Part | Default (OFFLINE) | `A365_LIVE=true` **and** `A365_ALLOW_WRITES=true` |
|---|---|---|
| Blueprint and 2 agent identities | **Fake.** Made-up IDs in memory; **nothing is created in Entra** | **Real.** Created in your tenant; the last cell deletes them |
| Access tokens | **Fake.** Unsigned tokens with the same fields a real one has | **Real.** Issued by Entra |
| Graph replies (user list, or 403) | **Fake.** 3 sample users for the researcher; 403 for the writer | **Real.** Your directory |
| LLM (`gpt-5.6-luna`) | **Real** when `AZURE_OPENAI_*` is set in `.env`; otherwise a stand-in text | same |

## 0. Setup

In [ ]:
# %pip install -q langgraph langchain-openai azure-identity requests

In [ ]:
import sys
from pathlib import Path

_here = Path(
    globals().get("__vsc_ipynb_file__", Path.cwd() / "_")
).parent  # make lab_helpers.py importable
sys.path[:0] = [str(_here), str(Path.cwd() / "Agent_365")]

from lab_helpers import load_settings, show_table  # noqa: E402

settings = load_settings()
REAL_IDENTITIES = (
    settings.live and settings.allow_writes
)  # creating identities is a change to your tenant
show_table(
    [
        {
            "part": "agent identities, tokens, Graph",
            "mode": "REAL (your tenant)" if REAL_IDENTITIES else "FAKE (nothing is created)",
        },
        {
            "part": "LLM",
            "mode": f"real: {settings.aoai_deployment}"
            if settings.llm_configured
            else "stand-in text",
        },
    ],
    title="This run",
)

## 0.1 Start here: which app, which agent, which token?

**Read sections 0.1-0.7 before running the live cells.** These are explanations, not commands: they do not sign you in, create identities, grant permissions, or call an MCP server.

Think of a school: an **ID number** tells us which person you mean; a **signed visitor pass** proves who arrived; the **door rules** decide where they may go.

- **Identification:** `appId` is a public identifier, not a password or a permission.
- **Authentication:** prove the caller's identity and obtain a signed token.
- **Authorization:** the receiving service checks whether that caller may perform this action on this data.
- **Agent 365** supplies management, governance, observability and managed tooling. **Microsoft Entra** issues the identity tokens. **MCP** is the tool protocol, not a replacement for either one.

| Name in this lab | What it actually is | Who creates it? | When is it used? |
|---|---|---|---|
| `settings.client_id` / `A365_CLIENT_ID` | The notebook's ordinary **setup/sign-in app registration** | You or an admin, once; reuse it if it already exists | `LiveGraph` uses it to sign in and provision the team |
| `BLUEPRINT['appId']` | The agent identity blueprint's application/client ID | `provision_agent_team()` creates or reuses it in LIVE mode | Authenticates the blueprint when asking Entra for an agent's token |
| `BLUEPRINT['principalId']` | The blueprint's **service principal object ID** in this tenant | The helper explicitly creates or reuses the blueprint principal | Makes the blueprint usable in this tenant; not a replacement for its appId |
| `AGENTS[role]['appId']` | The researcher or writer's **agent identity client ID** | The helper creates or reuses a service-principal-only agent identity | Selects that worker in `fmi_path` and the second token request |
| `AGENTS[role]['id']` | That worker's **service principal object ID** | Entra returns it with the agent identity in LIVE mode | Assign permissions to that object; manage or delete it |
| `00000003-0000-0000-c000-000000000000` | Microsoft Graph's fixed application ID | Microsoft, not this notebook | Find Graph's permission definitions; do not put it in `A365_CLIENT_ID` |
| Agent 365 registration ID / `agentCard` | A discoverability and management record describing an agent | Demonstrated separately in the showcase notebook | Find its metadata and endpoint; **not a credential or token** |

**Entra Agent ID exception:** for a worker **agent identity**, Microsoft documents that `appId` and `id` have the **same value**. They still name different property roles: client ID in token requests, object ID in directory operations. Do not generalize that equality to the setup app or blueprint, whose application/client and object IDs differ.

**The answer to "do I create the appId?":** create or reuse **one setup app registration**, then copy its **Application (client) ID** to `A365_CLIENT_ID`. Entra assigns the ID; you do not invent it. The notebook creates/reuses the blueprint and worker identities later. They are different objects with different IDs.

A LangGraph node name such as `researcher` does **not** automatically create an Entra identity. Our code deliberately connects that node to `AGENTS['researcher']`. The supervisor only routes Python state, so this example does not provision an identity for it.

## 0.2 Setup flow: how the notebook gets permission to create agents

This is the **provisioning flow**: an authorized person sets up the identities. It is not the researcher's later tool call.

```text
You/admin create or reuse contoso-agent365-notebook
    -> its client ID goes into A365_CLIENT_ID
    -> configure public-client sign-in + required delegated Graph permissions
    -> an authorized admin grants consent

Notebook: LiveGraph(settings, scopes=[...])
    -> InteractiveBrowserCredential signs YOU in through that setup app
    -> Entra issues a delegated token FOR Microsoft Graph
    -> Graph checks the app's consent AND your authority for the operation
    -> provision_agent_team() creates/reuses:
         blueprint application -> blueprint service principal
                                   -> researcher agent identity
                                   -> writer agent identity
    -> grant Graph User.Read.All to the RESEARCHER identity only
```

**Before LIVE mode:** follow [LIVE_SETUP.md, steps 1-5](./LIVE_SETUP.md), including the LangGraph-specific permissions in section 1 below. Save the tenant ID, setup client ID and owner/sponsor's object ID in the local `.env`. Configure `http://localhost` and public-client flows as the setup guide describes. Interactive sign-in needs no setup-app secret.

The selected PowerShell block in section 1 **adds three permissions to an existing setup app**. It does not create that app, the blueprint, or the workers. Those three permissions are additional to the base Agent ID write permissions. Admin consent alone does not give the signed-in person every required directory role.

**Separate permission grants:** setup-app permissions let the notebook **create/manage identities**. The researcher's `User.Read.All` application permission lets that worker **read directory users at runtime**. One grant does not automatically grant the other.

If `A365_CLIENT_SECRET` is configured, `LiveGraph` instead uses `ClientSecretCredential` and **application permissions** for the setup app. That is a different setup mode; the delegated-scope snippet alone does not configure it.

**Safety switch:** this notebook creates real identities only when **both** `A365_LIVE=true` and `A365_ALLOW_WRITES=true`. Otherwise the team IDs, worker tokens and Graph replies are simulated. These switches do not grant access; the real service still enforces permissions.

**Use disposable test identities:** the helper can reuse objects with the demo names and does not remove old permission grants. A reused writer with earlier grants might not be denied. The existing cleanup cell deletes the returned team objects, including reused ones; do not run it against a team you intend to keep.

## 0.3 Runtime flow: the researcher calls Graph as itself

Now the team already exists. The **runtime** uses the blueprint's credential to obtain a pass for one worker. It does not send the setup app's powerful provisioning token to the tool.

```text
researcher node -> graph_get_as('researcher', '/v1.0/users...')
    -> tokens.token_for(AGENTS['researcher']['appId'])

    1. Runtime -> Entra token endpoint
       client_id = BLUEPRINT['appId']
       blueprint credential + fmi_path = researcher's appId
       scope = api://AzureADTokenExchange/.default
       <- intermediate parent token (NOT a Graph access token)

    2. Runtime -> Entra token endpoint
       client_id = researcher's appId
       client_assertion = parent token
       scope = https://graph.microsoft.com/.default
       <- access token FOR Graph, issued for the researcher

    3. Runtime -> Microsoft Graph, Authorization: Bearer <Graph token>
       Graph validates the token and checks the operation's permissions
       <- users if authorized; denial if not
```

**The same route for the writer uses the writer's appId**, not the researcher's. The lab grants the researcher `User.Read.All` but gives the writer no Graph application roles, so the writer's user-list request is expected to fail.

Both exchanges here use `client_credentials`, and the final token is an **autonomous/app-only Graph token**, with no end user's delegated identity. `/.default` requests the permissions already granted for that resource; it does **not** mean "grant every permission". This lab assigns permissions directly to each worker; other supported Agent ID configurations can use permission inheritance through the blueprint.

**What about acting for a user?** An on-behalf-of (**OBO**) flow also needs a signed-in user's token addressed to the receiving agent/API, the appropriate delegated grants and the supported exchange for that service. Effective access is constrained by those grants, the user's access and service policy. Changing the prompt to "act as this user" is not OBO. This notebook does not implement it.

**Credential location:** the one-day demo secret belongs to the **blueprint**, not to either worker. For production, prefer a managed identity or workload identity with a federated trust on the blueprint. Never print the secret or full bearer tokens.

**Important limit:** both workers share one Python process and the token helper. Separate API identities demonstrate least privilege at the API boundary; they are not process isolation against malicious Python code. The writer can also read findings intentionally passed to it in shared state.

## 0.4 Where MCP fits: the agent is the caller, the server is the door

**MCP = Model Context Protocol.** The agent runtime contains an **MCP client**; the **MCP server** offers tools such as searching documents. A model choosing a tool does not give it permission to use that tool.

```text
Agent runtime / MCP client
    | token or other credential for the RECEIVING MCP service
    v
MCP server / Agent 365 gateway
    | authenticate caller; authorize tool; apply policy
    | use the server's documented downstream identity/delegation flow
    v
Graph / SharePoint / another API
      independently enforce that resource's access rules
```

**Does Agent ID belong to the agent or MCP?** In this lab it belongs to the **calling worker**. A remote MCP service is a separate API/resource, with its own authentication configuration and possibly its own application/workload identity for downstream calls. MCP itself does not create an Entra identity. The worker's appId alone is not proof of identity.

| Boundary | What must be checked? | What does NOT prove access? |
|---|---|---|
| Caller -> OAuth-protected MCP service | Valid token for this service: signature, trusted issuer/tenant, audience, expiry, required scopes/roles and policy | A worker name, appId argument, registry entry or Graph token with the wrong audience |
| MCP service -> data API | The supported downstream credential/delegation and the data API's permissions and resource rules | Merely being able to call the MCP server |
| Directory search -> model ranking | Filter by caller authorization **before** exposing candidates to a ranker/model | Asking the LLM to hide forbidden results after it already saw them |

### Microsoft-managed Agent 365 tools versus a custom MCP server

- **Managed tools:** use the server's documented audience, scopes and identity flow. Deployments can use a shared **Agent 365 Tools** resource with server-specific scopes (for example, `McpServers.Mail.All`) or **per-server resources** with `Tools.ListInvoke.All`. Read the actual configuration; do not assume one resource ID for every server.
- **Supported user-context flows:** Agent 365 tooling documents **OBO** (acting for a human user) and **agentic-user** access (acting as an associated agent-owned user account). The latter is not a normal app-only token; the identity documentation currently marks agent-owned users as Frontier preview. The two-step **app-only Graph demo here does not prove that a managed MCP server accepts the same flow**.
- **Custom remote MCP:** its operator chooses the authentication mechanism. Agent 365 bring-your-own registration documents Entra OAuth, external OAuth, API keys and no-auth options. A custom Entra-protected server can expose its own audience and scopes. The server must enforce its own policy; registration does not automatically secure direct calls that bypass the integration.
- **Downstream data:** an MCP server may use a supported delegated flow, an agentic-user flow or its own workload identity, depending on its design. Do not blindly forward the incoming bearer token to another audience. Follow that server's documentation; it does not necessarily require you to add a separate Graph grant to the notebook app.

**What this notebook actually does:** `graph_get_as()` is a Python function that calls Graph directly. There is **no live MCP connection** here. The showcase's local tooling manifest demonstrates discovery, not a completed live authentication or approval flow.

## 0.5 How does an admin approve MCP, and how does the agent know?

**Approval is saved configuration and consent, not a message sent to the LLM.** Keep these three decisions separate:

1. **Availability/governance:** is this MCP server allowed through the organization's Agent 365 integration?
2. **Identity and consent:** does this caller have the required permissions and supported sign-in/connection?
3. **Runtime authorization:** may this caller execute this tool against this data right now?

For the documented **bring-your-own remote MCP** workflow:

```text
Developer registers server endpoint + tools + authentication configuration
    -> admin: Microsoft 365 admin center > Agents > Tools > Requests
    -> review and approve/reject; consent to required Entra permissions
    -> approved server becomes available in supported clients
    -> connection setup may still be required (for example, an API key)
    -> each invocation is subject to service policy and authorization
    -> an admin can later block the server; the integration enforces the block
```

For **pro-code managed tooling**, the manifest/SDK configuration identifies the tool servers, audiences and scopes. Consent/setup is a separate prerequisite. Adding a line to `ToolingManifest.json` does **not** grant access.

**How the runtime learns:** discovery/configuration tells the MCP client which servers to connect to; the identity library acquires the required token; the gateway/server enforces the applicable controls. The LLM may see an allowed tool list, but it is not the authority that approves access. Availability and consent changes can take time to propagate.

**Limits to remember:** the documented BYO controls govern integrated traffic, not every independent connection to the original endpoint. BYO is preview, and supported clients vary; registration alone does not mean any Foundry or LangGraph app is automatically connected.

**Example:** an admin approving a mail MCP server does not give the writer `User.Read.All`, permission to every mailbox or permission to call the model. Those are different resources and authorization decisions. An `agentCard`, Entra tag or observability span is descriptive data, not an access grant.

## 0.6 Three doors: Graph, MCP and the model

A school-library pass is not a science-lab pass. Likewise, an OAuth access token has an **audience** (`aud`): the API it was issued for. Depending on token version, `aud` can be a resource URI or its application ID.

| Call | Identity in this notebook | Requested resource/scope | Where permission comes from |
|---|---|---|---|
| Notebook -> Graph to provision the team | You through the setup app (or the setup app alone in secret mode) | Explicit Graph delegated scopes; app-only setup uses Graph `/.default` | Setup-app consent plus required user authority in delegated mode |
| Researcher/writer -> Graph for a tool | That worker's Entra Agent ID | `https://graph.microsoft.com/.default` | The worker's Graph application-role assignment in this lab |
| Runtime -> managed MCP service | **Conceptual only here**; use the service's supported user/agent identity flow | Its configured Agent 365 Tools **or per-server** resource/scopes | Tool consent, supported connection and service policy |
| Runtime -> custom remote MCP | **Conceptual only here**; depends on the server's authentication contract | That server's audience/scopes, or another configured mechanism | Its operator's access policy and grants |
| LangGraph -> Azure OpenAI model | Whatever `DefaultAzureCredential` selects, often your developer login | `https://cognitiveservices.azure.com/.default` | Model-resource access, e.g. Cognitive Services OpenAI User |

**The model does not automatically run as the researcher.** In section 4 the model client is shared and has its own credential provider. Giving a worker a Graph permission changes neither that provider nor the model's RBAC.

Two unrelated meanings of **token**:
- **Access token:** a secret bearer credential issued by Entra. Do not print or paste it into chat, a slide, source control or an online decoder.
- **LLM token:** a piece of text counted for context length and billing. An output such as `{'input': 13, 'output': 7}` counts text pieces; it says nothing about sign-in or permissions.

The existing `decode_jwt_claims()` helper only displays selected claims for teaching. **Decoding is not validation.** A real API must verify the token and enforce authorization; the offline unsigned samples must never be accepted as real credentials.

## 0.7 Follow the code, then check your understanding

| Look for this code below | What it proves |
|---|---|
| `LiveGraph(settings, scopes=[...])` | The **setup app** signs in for Graph provisioning |
| `provision_agent_team(...)` | Create/reuse blueprint and worker identities; grant the researcher access |
| `AgentTokens.token_for(agent_app_id)` | Blueprint-to-worker exchange for a **Graph** token, not MCP/model access |
| `graph_get_as(role, path)` | Send the selected worker's token to Graph; record the response |
| `get_bearer_token_provider(DefaultAzureCredential(), ...)` | Separate sign-in and resource audience for the **model** |
| `team.invoke(...)` | Run the local LangGraph workflow; it does not deploy Foundry-hosted agents |

**Troubleshoot the correct boundary:**

| Symptom | First checks |
|---|---|
| Setup cannot sign in / admin approval required | Correct tenant and **setup client ID**; redirect URI/public-client settings; requested scopes and consent |
| Token cannot be obtained for a worker | Blueprint credential/trust, blueprint principal, correct worker ID and tenant; inspect Entra's error without printing secrets |
| API returns 401 or rejects token | Intended audience, trusted issuer/tenant, signature and expiry; status alone is not a complete diagnosis |
| API returns 403 | Relevant caller's grants, consent, resource/user access and policy; a valid identity is not unlimited authorization |
| MCP server is listed but its call fails | Server approval/block state, actual authentication configuration, consent and downstream access; listing is not authorization |
| Graph works but the model call fails | The **model credential's** tenant/resource RBAC and model endpoint; worker Graph roles do not fix model access |

**Quick check:**
1. Which app do I set up first? **The notebook setup app; its client ID is `A365_CLIENT_ID`.**
2. Who creates the researcher and writer identities? **The helper, only in live/write mode; otherwise they are fake.**
3. Is the researcher's Graph token a universal MCP pass? **No: audience and supported flow must match the receiving service.**
4. Is a registered/approved tool automatically allowed to read all data? **No: consent, caller rights and resource policy still apply.**
5. Does this notebook prove live MCP or user-OBO works? **No: it demonstrates direct Graph access with app-only worker identities.**

### Sources and scope

Documentation checked for this walkthrough on **28 September 2026**. Preview APIs, client support and admin UI can change. The examples use Contoso and contain no real tokens or secrets.

- [Entra agent identities: object model and appId/id equality](https://learn.microsoft.com/entra/agent-id/agent-identities)
- [Blueprints and blueprint principals](https://learn.microsoft.com/entra/agent-id/agent-blueprint)
- [Autonomous agent authentication and authorization](https://learn.microsoft.com/entra/agent-id/autonomous-agent-authentication-authorization-flow)
- [Agent 365 identity and execution modes](https://learn.microsoft.com/microsoft-agent-365/developer/identity)
- [Agent 365 tooling: manifest, scopes and OBO](https://learn.microsoft.com/microsoft-agent-365/developer/tooling)
- [Admin approval of remote MCP servers](https://learn.microsoft.com/microsoft-365/admin/manage/manage-tools-for-agent)
- [Graph delegated and application access](https://learn.microsoft.com/graph/auth/auth-concepts)
- [First-party tooling resource provisioning: shared and per-server resources](https://github.com/microsoft/Agent365-devTools/blob/16e3d1e54c3e70648edd7fc1912275bb5f1fd329/scripts/cli/Auth/New-Agent365ToolsServicePrincipalProdPublic.ps1)

**Next:** section 1 provisions the team. Keep LIVE/write mode off while learning unless you intend to modify a test tenant.

## 1. Create the team's identities

We create **one blueprint** ("the contoso LangGraph team") and **one agent identity per worker**, then give **only the
researcher** permission to read users. In LIVE mode these are plain Microsoft Graph calls (code in `lab_helpers.py`):

| # | Graph call | Why |
|---|---|---|
| 1 | `POST /applications/microsoft.graph.agentIdentityBlueprint` | create the blueprint (the template, which holds the credential) |
| 2 | `POST /servicePrincipals/microsoft.graph.agentIdentityBlueprintPrincipal` | make the blueprint usable in this tenant. **Not automatic**; skipping it causes errors |
| 3 | `POST /applications/{id}/microsoft.graph.agentIdentityBlueprint/addPassword` | a **1-day** secret for this demo (production: use a managed identity instead) |
| 4 | `POST /servicePrincipals/microsoft.graph.agentIdentity` (×2) | one agent identity for `researcher`, one for `writer` |
| 5 | `POST /servicePrincipals/{researcher}/appRoleAssignments` | grant `User.Read.All` to the **researcher only** |

**LIVE only, one-time:** the existing **setup/sign-in app** (`A365_CLIENT_ID`, not a worker or blueprint ID) needs these 3 extra delegated permissions after the base setup in [LIVE_SETUP.md](./LIVE_SETUP.md). This block adds permissions; it does not create an app. Run it in PowerShell with an account authorized to grant admin consent:
```powershell
$appId = "<A365_CLIENT_ID>"; $graphAppId = "00000003-0000-0000-c000-000000000000"
$graph = az ad sp show --id $graphAppId -o json | ConvertFrom-Json
foreach ($n in "AgentIdentityBlueprintPrincipal.ReadWrite.All", "AppRoleAssignment.ReadWrite.All", "Application.Read.All") {
    $s = $graph.oauth2PermissionScopes | Where-Object value -eq $n
    az ad app permission add --id $appId --api $graphAppId --api-permissions "$($s.id)=Scope" --only-show-errors }
Start-Sleep 30; az ad app permission admin-consent --id $appId
```

In [ ]:
from lab_helpers import LiveGraph, provision_agent_team

WORKERS = {
    "researcher": "contoso-lg-researcher",
    "writer": "contoso-lg-writer",
}  # role → agent identity name
PERMISSIONS = {
    "researcher": ["User.Read.All"],
    "writer": [],
}  # least privilege: the writer gets nothing

# This uses the SETUP app (settings.client_id / A365_CLIENT_ID), not a worker.
# Without A365_CLIENT_SECRET, LiveGraph opens a browser for your sign-in.
# The scopes below authorize provisioning; they are not worker tool permissions.
if REAL_IDENTITIES:
    admin = LiveGraph(
        settings,
        scopes=[
            "AgentIdentityBlueprint.ReadWrite.All",
            "AgentIdentityBlueprintPrincipal.ReadWrite.All",
            "AgentIdentity.ReadWrite.All",
            "AppRoleAssignment.ReadWrite.All",
            "Application.Read.All",
        ],
    )
    # Entra assigns the IDs. The helper creates/reuses the blueprint and its
    # principal, then worker service principals and their individual grants.
    # A worker is NOT a new ordinary app registration with its own password.
    BLUEPRINT, AGENTS = provision_agent_team(
        admin, "contoso-langgraph-team-blueprint", WORKERS, PERMISSIONS, settings.owner_id
    )
else:  # FAKE identities: just made-up IDs
    BLUEPRINT = {"appId": "b1000000-0000-4000-8000-00000000b001", "secret": "(none: fake)"}
    AGENTS = {
        "researcher": {
            "appId": "a1000000-0000-4000-8000-000000000001",
            "name": WORKERS["researcher"],
        },
        "writer": {"appId": "a1000000-0000-4000-8000-000000000002", "name": WORKERS["writer"]},
    }

MODE_LABEL = "REAL, created in your tenant" if REAL_IDENTITIES else "FAKE, nothing created in Entra"
show_table(
    [
        {
            "role": role,
            "agent identity": a["name"],
            "appId": a["appId"],
            "permissions": ", ".join(PERMISSIONS[role]) or "none",
        }
        for role, a in AGENTS.items()
    ],
    title=f"Team identities ({MODE_LABEL})",
)

## 2. How an agent gets its own token

**Analogy:** the blueprint is the **badge office** and holds the master key. An agent identity is an **employee**. The
employee has no key of its own, so:

| Step | Who asks | What they send | What they get |
|---|---|---|---|
| 1 | the **blueprint** | its secret + "`fmi_path` = *this agent's* appId" | a short "parent" token that is valid only for **that one agent** |
| 2 | the **agent** | its own appId + the parent token | an **access token for Graph that says "I am the researcher"**, listing the researcher's permissions |

Both steps are ordinary requests to Entra's token endpoint (`https://login.microsoftonline.com/{tenant}/oauth2/v2.0/token`).
For **these two app-only exchanges**, the scope ends in `/.default`, and `fmi_path` uses the worker's **appId** property. For an Entra agent identity, that value equals its object ID; do not substitute the setup app or blueprint ID. Other flows, such as interactive sign-in, can request explicit delegated scopes.

The code below obtains **Graph tokens only**. It does not obtain a token for an MCP server or for the model. See section 0.6 for those separate audiences.

In [ ]:
import time

import requests
from lab_helpers import FakeAgentTokens, decode_jwt_claims

TOKEN_URL = f"https://login.microsoftonline.com/{settings.tenant_id}/oauth2/v2.0/token"


class AgentTokens:
    """Real tokens: the two-step exchange from the table above, cached until they expire."""

    def __init__(self, blueprint_app_id: str, blueprint_secret: str):
        self.blueprint_app_id, self.blueprint_secret, self.cache = (
            blueprint_app_id,
            blueprint_secret,
            {},
        )

    def token_for(self, agent_app_id: str) -> str:
        if agent_app_id in self.cache and self.cache[agent_app_id][1] > time.time() + 60:
            return self.cache[agent_app_id][0]
        # Step 1: authenticate the BLUEPRINT and identify the target worker.
        # This is a token request to Entra, not an agent-creation request.
        # The resulting parent token is only an intermediate credential.
        parent = self._request(
            {
                "grant_type": "client_credentials",
                "client_id": self.blueprint_app_id,
                "client_secret": self.blueprint_secret,
                "fmi_path": agent_app_id,
                "scope": "api://AzureADTokenExchange/.default",
            }
        )["access_token"]
        # Step 2: use that intermediate credential to authenticate THIS worker.
        # The resource is Graph. An MCP server needs its own supported audience.
        # /.default uses existing grants; it does not add permissions.
        reply = self._request(
            {
                "grant_type": "client_credentials",
                "client_id": agent_app_id,
                "client_assertion_type": "urn:ietf:params:oauth:client-assertion-type:jwt-bearer",
                "client_assertion": parent,
                "scope": "https://graph.microsoft.com/.default",
            }
        )
        self.cache[agent_app_id] = (reply["access_token"], time.time() + reply["expires_in"])
        return reply["access_token"]

    def clear(self):
        self.cache.clear()

    @staticmethod
    def _request(form: dict) -> dict:
        for _ in range(5):  # brand-new identities can take ~30 s to become usable, so retry
            reply = requests.post(TOKEN_URL, data=form, timeout=15)
            if reply.ok:
                return reply.json()
            time.sleep(10)
        raise RuntimeError(reply.json().get("error_description", reply.text)[:300])


if REAL_IDENTITIES:
    tokens = AgentTokens(BLUEPRINT["appId"], BLUEPRINT["secret"])
else:
    tokens = FakeAgentTokens(
        BLUEPRINT["appId"], {a["appId"]: PERMISSIONS[r] for r, a in AGENTS.items()}
    )

rows = []
for role, agent in AGENTS.items():
    # Decoding is for this teaching table ONLY. It does not verify a signature.
    # The receiving API must validate the real token; never trust decoded claims alone.
    claims = decode_jwt_claims(tokens.token_for(agent["appId"]))
    rows.append(
        {
            "agent": role,
            "appid (who is calling)": claims.get("appid"),
            "roles (what it may do)": ", ".join(claims.get("roles", [])) or "none",
            "xms_par_app_azp (its blueprint)": claims.get("xms_par_app_azp"),
        }
    )
show_table(
    rows, title=f"What's inside each agent's token ({'real' if REAL_IDENTITIES else 'FAKE'} tokens)"
)

**What to notice:** both tokens come from the **same blueprint**, but each says a **different "who"** (`appid`), and only the
researcher's token lists `User.Read.All`. Graph validates the token (including its signature, issuer, audience and expiry) and applies its authorization rules. The notebook's claim decoder is not that validator.

## 3. The tool the agents use: "call Graph as yourself"

`graph_get_as(agent, path)` gets that agent's token and calls Graph with it. Every call is written to an **audit list**, so at
the end we can show *who called what, with which permissions, and what happened*.

**This tool is a local Python function making a direct HTTP call to Graph. There is no MCP client, MCP server or Agent 365 tooling gateway in this call path.** Sections 0.4-0.5 explain where those components would fit.

In [ ]:
from lab_helpers import fake_graph_reply

AUDIT = []


def graph_get_as(role: str, path: str) -> tuple[int, dict]:
    # role selects the worker identity. This is NOT the setup app's token.
    # The live request below sends that worker's Graph token to Graph only.
    token = tokens.token_for(AGENTS[role]["appId"])
    claims = decode_jwt_claims(token)
    if REAL_IDENTITIES:
        reply = requests.get(
            "https://graph.microsoft.com" + path,
            headers={"Authorization": f"Bearer {token}"},
            timeout=30,
        )
        status, body = reply.status_code, (reply.json() if reply.content else {})
    else:
        status, body = fake_graph_reply(claims, path)
    AUDIT.append(
        {
            "agent": role,
            "calling as (appid)": claims.get("appid"),
            "permissions in token": ", ".join(claims.get("roles", [])) or "none",
            "request": "GET " + path.split("?")[0],
            "result": "✅ 200 OK" if status == 200 else f"🚫 {status}",
        }
    )
    return status, body

## 4. The LangGraph team

| Node (agent) | What it does | Identity used |
|---|---|---|
| **supervisor** | decides who works next: researcher first, then writer, then stop. (Plain Python here; you could let an LLM decide.) | none: it only routes |
| **researcher** | calls Graph **as the researcher** to list users, then asks the LLM to turn them into a short "who's who" | `contoso-lg-researcher` (has `User.Read.All`) |
| **writer** | **tries** the same Graph call as the writer (expected: refused), then asks the LLM to write the note from the researcher's findings | `contoso-lg-writer` (no permissions) |

The **state** is the dictionary the agents share: `task` (the request), `findings` (from the researcher), `draft` (from the
writer), and `steps` (the order the nodes ran in).

The LLM uses a **separate `DefaultAzureCredential` chain**, which may select your Azure CLI login or another configured credential. It requests a token for Azure AI Services, not Graph, and needs access to the model resource. The **Graph calls** use the **agents' own identities**. A LangGraph node name does not automatically change the LLM credential.

In [ ]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph

if settings.llm_configured:
    from azure.identity import DefaultAzureCredential, get_bearer_token_provider
    from langchain_openai import AzureChatOpenAI

    chat_model = AzureChatOpenAI(
        azure_endpoint=settings.aoai_endpoint,
        azure_deployment=settings.aoai_deployment,
        api_version=settings.aoai_api_version,
        # Separate door, separate pass: this token is for the MODEL service.
        # DefaultAzureCredential selects an available developer/host credential;
        # it is not automatically the researcher or writer Agent ID.
        azure_ad_token_provider=get_bearer_token_provider(
            DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
        ),
    )


def ask_llm(prompt: str, stand_in: str) -> str:
    return chat_model.invoke(prompt).content if settings.llm_configured else stand_in


class TeamState(TypedDict, total=False):
    task: str
    findings: str
    draft: str
    steps: Annotated[list[str], operator.add]  # each node appends its name


def supervisor(state: TeamState) -> dict:
    return {"steps": ["supervisor"]}


def who_is_next(state: TeamState) -> str:
    if not state.get("findings"):
        return "researcher"
    if not state.get("draft"):
        return "writer"
    return END


def researcher(state: TeamState) -> dict:
    status, body = graph_get_as("researcher", "/v1.0/users?$top=5&$select=displayName,jobTitle")
    people = [
        f"{u.get('displayName')} ({u.get('jobTitle') or 'no title'})" for u in body.get("value", [])
    ]
    stand_in = "\n".join(f"- {p}" for p in people) or f"(directory lookup failed: {status})"
    findings = ask_llm(f"List who's who as short factual bullets (no greeting): {people}", stand_in)
    return {"findings": findings, "steps": ["researcher"]}


def writer(state: TeamState) -> dict:
    graph_get_as("writer", "/v1.0/users?$top=1")  # the writer has no permission: expect 403
    stand_in = f"Welcome to the team! People to meet:\n{state['findings']}"
    draft = ask_llm(
        f"Task: {state['task']}\nWrite a 3-sentence welcome note introducing colleagues by role only: "
        f"these people:\n{state['findings']}",
        stand_in,
    )
    return {"draft": draft, "steps": ["writer"]}


graph = StateGraph(TeamState)
graph.add_node("supervisor", supervisor)
graph.add_node("researcher", researcher)
graph.add_node("writer", writer)
graph.add_edge(START, "supervisor")  # always start at the supervisor
graph.add_conditional_edges("supervisor", who_is_next, ["researcher", "writer", END])
graph.add_edge("researcher", "supervisor")  # workers report back to the supervisor
graph.add_edge("writer", "supervisor")
team = graph.compile()
print("Team ready:", " / ".join(n for n in team.get_graph().nodes if not n.startswith("__")))

## 5. Run the team

In [ ]:
AUDIT.clear()
tokens.clear()  # fetch fresh tokens, so permissions granted a minute ago are included
result = team.invoke(
    {
        "task": "Welcome a new team member and introduce colleagues by role.",
        "steps": [],
    }
)

print("Order the agents ran in:", " → ".join(result["steps"]))
print("\nResearcher's findings:\n" + result["findings"])
print("\nWriter's draft:\n" + result["draft"])
replies = "real" if REAL_IDENTITIES else "FAKE"
show_table(AUDIT, title=f"Audit: who called Graph, as which identity ({replies} replies)")

### What to notice
* **Same request, different outcome.** Both agents asked Graph for the user list. The researcher got it; the writer was
  refused, even though both run in the same Python process.
* **The logs name the agent.** Each call carries that agent's `appid`. In LIVE mode, Entra's sign-in logs show
  `contoso-lg-researcher` or `contoso-lg-writer`, not one shared app.
* **The writer still did its job.** It used the researcher's findings from the shared state and never needed directory access.

> LIVE tip: straight after the identities are created, the researcher may also get a 403 for 1–2 minutes while its
> permission takes effect. Just run the cell again.

**Going to production**
* Replace the blueprint secret with a **managed identity** (set as a federated credential on the blueprint): no secrets to leak.
* For Python services, the [Microsoft Entra SDK for AgentID](https://learn.microsoft.com/entra/msidweb/agent-id-sdk/overview)
  runs as a side-car and does the token exchange for you.
* To act **on behalf of a signed-in user**, use the *OBO* variant of step 2.
* Register the agents in Agent 365 (see the first notebook, section 5.3) so admins can find and govern them.

## 6. Clean up

In [ ]:
from lab_helpers import delete_agent_team

if REAL_IDENTITIES:
    delete_agent_team(admin, BLUEPRINT, AGENTS)
    print("Deleted the 2 agent identities, the blueprint principal and the blueprint.")
else:
    print("Nothing to delete: the identities were fake.")